# Reprojection In Depth

The `Reprojector` class wraps `rasterio.warp.reproject` with
sensible defaults and automatic caching. This notebook covers:
- Changing CRS with auto-resolved extent
- Selecting bands
- Applying scale/offset during warp
- GeoTIFF creation options
- NetCDF subdatasets

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import numpy as np
from rasterio.enums import Resampling
from pyproj import CRS
from affine import Affine

from pygeodata import Data, SpatialSpec, load, setconfig
from pygeodata.processors.reprojection import Reprojector
from pygeodata.options import RasterCreationOptions

setconfig(path_data_processed=Path("./data/processed"))

## 1. Reproject to a different CRS — auto-resolve extent

When `spec` is not fully defined (no `transform`/`shape`),
`Reprojector.resolve_spec()` infers them from the source file.

In [ ]:
spec_moll = SpatialSpec(crs=CRS.from_proj4("+proj=moll +lon_0=0 +datum=WGS84"))

@dataclass
class DEMMollweide(Data):
    src: str = "data/raw/dem.tif"

    @property
    def processor(self):
        return Reprojector(srcpath=self.src, resampling=Resampling.bilinear)


loader = DEMMollweide()
resolved = loader.resolve_spec(spec_moll)
print("Resolved shape:    ", resolved.shape)
print("Resolved transform:", resolved.transform)

## 2. Reproject a subset of bands

In [ ]:
@dataclass
class Band1Loader(Data):
    """Extract and reproject only band 1 from a multi-band raster."""
    src: str = "data/raw/multispectral.tif"

    @property
    def processor(self):
        return Reprojector(
            srcpath=self.src,
            bands=1,                       # single band (1-indexed)
            resampling=Resampling.nearest,
        )

## 3. Apply scale/offset during warp

`scales` and `offsets` apply `y = scale * x + offset` to pixel values
without a separate processing step — useful for DN → physical unit conversions.

In [ ]:
@dataclass
class ScaledTemperatureLoader(Data):
    """Convert raw DN values to Kelvin during reprojection."""
    src: str = "data/raw/lst.tif"
    scale_factor: float = 0.02

    @property
    def processor(self):
        return Reprojector(
            srcpath=self.src,
            scales=self.scale_factor,
            resampling=Resampling.bilinear,
            dst_dtype=np.float32,
            dst_nodata=np.nan,
        )

## 4. Custom GeoTIFF creation options

Control compression, tiling, and blocksize via `RasterCreationOptions`.

In [ ]:
opts = RasterCreationOptions(compress="lzw", tiled=True, blockxsize=256, blockysize=256)

@dataclass
class CompressedDEMLoader(Data):
    src: str = "data/raw/dem.tif"

    @property
    def processor(self):
        return Reprojector(srcpath=self.src, raster_creation_options=opts)

## 5. NetCDF subdatasets

Use the `netcdf:file.nc:variable` path syntax. Set `force_read=True`
to avoid the empty-raster bug caused by `rio.band()` on multi-variable datasets.

In [ ]:
@dataclass
class ERA5PrecipLoader(Data):
    src: str = "netcdf:data/raw/era5.nc:tp"

    @property
    def processor(self):
        return Reprojector(
            srcpath=self.src,
            force_read=True,               # required for NetCDF subdatasets
            resampling=Resampling.bilinear,
            dst_nodata=np.nan,
        )

## 6. Override source CRS

For files that lack embedded CRS metadata, provide it explicitly.

In [ ]:
from rasterio import CRS as RioCRS

@dataclass
class LegacyDEMLoader(Data):
    src: str = "data/raw/legacy_dem_no_prj.tif"

    @property
    def processor(self):
        return Reprojector(
            srcpath=self.src,
            src_crs=RioCRS.from_epsg(32632),   # override missing CRS
            resampling=Resampling.bilinear,
        )